IMPORT LIBRARIES

In [4]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

Load Data & Inspect

In [5]:
data = pd.read_csv('final_data.csv')

print("--- First 10 Rows ---")
print(data.head(10))

print("\n--- DataFrame Info ---")
data.info()

df = data.copy()

--- First 10 Rows ---
   user_id          signup_time        purchase_time  purchase_value  \
0   247547  2015-06-28 03:00:34  2015-08-09 03:57:29              47   
1   220737  2015-01-28 14:21:11  2015-02-11 20:28:28              15   
2   390400  2015-03-19 20:49:09  2015-04-11 23:41:23              44   
3    69592  2015-02-24 06:11:57  2015-05-23 16:40:14              55   
4   174987  2015-07-07 12:58:11  2015-11-03 04:04:30              51   
5    23204  2015-06-15 21:47:39  2015-08-02 04:35:34              25   
6   155230  2015-03-15 08:27:21  2015-05-21 13:55:15              37   
7   199369  2015-01-17 14:38:23  2015-03-15 18:13:39              43   
8   236894  2015-06-16 04:02:41  2015-07-25 01:29:33              22   
9   379446  2015-04-11 13:13:27  2015-07-03 22:27:10              49   

       device_id  source browser sex  age    ip_address  class  \
0  KIXYSVCHIPQBR     SEO  Safari   F   30  1.677886e+07      0   
1  PKYOWQKWGJNJI     SEO  Chrome   F   34  1.684205e+

FEATURE ENGINEERING

Device ID Feature Engineering

In [6]:
device_counts = df.groupby('device_id')['user_id'].nunique()
df['device_id_user_counts'] = df['device_id'].map(device_counts)

import joblib
joblib.dump(device_counts, 'device_counts_map.joblib')

max_users = df['device_id_user_counts'].max()
print(f"The maximum number of unique users on a single device is: {max_users}")

df_sorted_by_device = df.sort_values('device_id_user_counts', ascending=False)

print("Top 10 devices by number of unique users:")
print(df_sorted_by_device.head(10))

print("\nDistribution of user counts per device:")
print(df['device_id_user_counts'].value_counts())

df.head()

The maximum number of unique users on a single device is: 20
Top 10 devices by number of unique users:
       user_id          signup_time        purchase_time  purchase_value  \
11331    65147  2015-01-04 15:02:04  2015-01-04 15:02:05              32   
51650   101698  2015-01-06 06:33:21  2015-01-06 06:33:22              47   
51641   257171  2015-01-06 06:33:18  2015-03-29 03:32:06              47   
51642    52655  2015-01-06 06:33:31  2015-01-06 06:33:32              47   
51643   216960  2015-01-06 06:33:33  2015-01-06 06:33:34              47   
51644     8863  2015-01-06 06:33:26  2015-01-06 06:33:27              47   
51645   308527  2015-01-06 06:33:29  2015-01-06 06:33:30              47   
51646   315519  2015-01-06 06:33:20  2015-01-06 06:33:21              47   
19529    74754  2015-01-03 10:47:48  2015-01-03 10:47:49              58   
51647   151070  2015-01-06 06:33:27  2015-01-06 06:33:28              47   

           device_id source browser sex  age    ip_address  

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,lower_bound_ip_address,upper_bound_ip_address,country,device_id_user_counts
0,247547,2015-06-28 03:00:34,2015-08-09 03:57:29,47,KIXYSVCHIPQBR,SEO,Safari,F,30,1.677886e+07,0,16778240.0,16779263.0,Australia,1
1,220737,2015-01-28 14:21:11,2015-02-11 20:28:28,15,PKYOWQKWGJNJI,SEO,Chrome,F,34,1.684205e+07,0,16809984.0,16842751.0,Thailand,1
2,390400,2015-03-19 20:49:09,2015-04-11 23:41:23,44,LVCSXLISZHVUO,Ads,IE,M,29,1.684366e+07,0,16843264.0,16843775.0,China,2
3,69592,2015-02-24 06:11:57,2015-05-23 16:40:14,55,UHAUHNXXUADJE,Direct,Chrome,F,30,1.693873e+07,0,16924672.0,16941055.0,China,1
4,174987,2015-07-07 12:58:11,2015-11-03 04:04:30,51,XPGPMOHIDRMGE,SEO,Chrome,F,37,1.697198e+07,0,16941056.0,16973823.0,Thailand,1


IP Address Column Cleanup

In [7]:
df.drop(['ip_address'],axis=1, inplace=True)
df.drop(['lower_bound_ip_address'],axis=1, inplace=True)
df.drop(['upper_bound_ip_address'],axis=1, inplace=True)

df.head()

fraud = df[df['class'] == 1]
valid = df[df['class'] == 0]
print('Total Cases: {}'.format(len(df['class'])))
print('Fraud Cases: {}'.format(len(df[df['class'] == 1])))
print('Valid Transactions: {}'.format(len(df[df['class'] == 0])))

Total Cases: 129146
Fraud Cases: 12268
Valid Transactions: 116878


Time Difference Feature Engineering

In [8]:
# EXTRACTING THE TIME DIFFERENCE BETWEEN SIGNUP AND PURCHASE TIME
df["signup_time"] = pd.to_datetime(df["signup_time"], format="%Y-%m-%d %H:%M:%S")
df["purchase_time"] = pd.to_datetime(df["purchase_time"], format="%Y-%m-%d %H:%M:%S")

df["time_diff_min"] = (df["purchase_time"] - df["signup_time"]).dt.total_seconds() / 60

df.head(20)

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,class,country,device_id_user_counts,time_diff_min
0,247547,2015-06-28 03:00:34,2015-08-09 03:57:29,47,KIXYSVCHIPQBR,SEO,Safari,F,30,0,Australia,1,60536.916667
1,220737,2015-01-28 14:21:11,2015-02-11 20:28:28,15,PKYOWQKWGJNJI,SEO,Chrome,F,34,0,Thailand,1,20527.283333
2,390400,2015-03-19 20:49:09,2015-04-11 23:41:23,44,LVCSXLISZHVUO,Ads,IE,M,29,0,China,2,33292.233333
3,69592,2015-02-24 06:11:57,2015-05-23 16:40:14,55,UHAUHNXXUADJE,Direct,Chrome,F,30,0,China,1,127348.283333
4,174987,2015-07-07 12:58:11,2015-11-03 04:04:30,51,XPGPMOHIDRMGE,SEO,Chrome,F,37,0,Thailand,1,170826.316667
5,23204,2015-06-15 21:47:39,2015-08-02 04:35:34,25,CSFNDSDBQATBA,Ads,IE,F,43,0,China,1,68087.916667
6,155230,2015-03-15 08:27:21,2015-05-21 13:55:15,37,BRDZLYDRFATRF,Ads,Chrome,F,46,0,Thailand,1,96807.900000
7,199369,2015-01-17 14:38:23,2015-03-15 18:13:39,43,MURSXNWVBRZVM,Direct,IE,M,37,0,Japan,2,82295.266667
8,236894,2015-06-16 04:02:41,2015-07-25 01:29:33,22,YXWLWLZBQVBMG,Ads,Chrome,M,31,0,Japan,1,56006.866667
9,379446,2015-04-11 13:13:27,2015-07-03 22:27:10,49,FAWCZLEWWUDVD,SEO,Chrome,M,24,0,Japan,2,120073.716667


Dropping Unused IDs

In [9]:
df.drop(['device_id'],axis=1, inplace=True)
df.drop(['user_id'],axis=1, inplace=True)

Datetime Feature Engineering

In [10]:
# EXTRACTING HOURLY, DAY OF THE WEEK, AND DAY IN A MONTH FEATURES
df["signup_hour"] = df["signup_time"].dt.hour
df["purchase_hour"] = df["purchase_time"].dt.hour
df["purchase_dayofweek"] = df["purchase_time"].dt.dayofweek
df["purchase_month"] = df["purchase_time"].dt.month 

print(pd.crosstab(df['purchase_hour'], df['class']))

print(pd.crosstab(df['purchase_dayofweek'], df['class']))

class             0    1
purchase_hour           
0              4782  502
1              4806  502
2              4934  522
3              5029  536
4              4747  442
5              4770  496
6              4986  499
7              4812  499
8              4872  563
9              4953  571
10             4697  519
11             4887  502
12             4910  509
13             4883  482
14             4944  521
15             4889  543
16             4870  558
17             4902  615
18             4861  486
19             4851  534
20             4826  433
21             4812  501
22             4975  460
23             4880  473
class                   0     1
purchase_dayofweek             
0                   16723  1842
1                   16789  1415
2                   16703  1456
3                   16608  1834
4                   16457  1917
5                   16681  1875
6                   16917  1929


Cyclic Time Representation

In [ ]:
# CYCLIC REPRESENTATION OF TIME
df['purchase_hour_sin'] = np.sin(2 * np.pi * df['purchase_hour']/24.0)
df['purchase_hour_cos'] = np.cos(2 * np.pi * df['purchase_hour']/24.0)
df['signup_hour_sin'] = np.sin(2 * np.pi * df['signup_hour']/24.0)
df['signup_hour_cos'] = np.cos(2 * np.pi * df['signup_hour']/24.0)

df.head()

Time Difference Category

In [ ]:
# TIME DIFFERENCE CATEGORY
bins = [0, 1, 10, 60, np.inf] 
labels = ['< 1 min', '1-10 mins', '10-60 mins', '> 60 mins']
df['time_diff_category'] = pd.cut(df['time_diff_min'], bins=bins, labels=labels, right=False)
df = df.drop('time_diff_min', axis=1)
print(df['time_diff_category'].value_counts())

joblib.dump(bins, 'time_bins.joblib')
joblib.dump(labels, 'time_labels.joblib')

crosstab_df = pd.crosstab(df['time_diff_category'], df['class'])
print(crosstab_df)